## Import Modules

In [ ]:
import matplotlib.pyplot as plt

from utils import *

plt.rcParams['figure.dpi'] = 110

DATASETS = ['statlog', 'chd', 'framingham', 'heart', 'stroke']
MODEL_NAMES = ['logreg', 'knn', 'decisiontree', 'svm', 'naivebayes',
               'randomforest', 'xgb', 'lightgbm']
TOP_N = 5

## Compute SHAP: Baseline

In [ ]:
baseline_shap = {}  # key: (ds, model_name) -> pd.Series (mean |SHAP|, top N)

for ds in DATASETS:
    df_train, scaler = get_preprocessed(ds, ds)
    X_train = df_train.drop(columns=['target'])

    for model_name in MODEL_NAMES:
        print(f'SHAP baseline: {ds} / {model_name}')
        model = load_baseline_model(ds, model_name)
        try:
            shap_vals = compute_shap(model, X_train)
            importance = mean_abs_shap(shap_vals, X_train.columns)
            baseline_shap[(ds, model_name)] = importance.head(TOP_N)
        except Exception as e:
            print(f'SKIP ({e})')
            baseline_shap[(ds, model_name)] = None

## Compute SHAP - Best Concat per Dataset

In [ ]:
df_eval = pd.read_csv('result/eval_concat.csv')

best_concat_cfg = (
    df_eval.loc[df_eval.groupby(['dataset', 'model'])['f1_macro'].idxmax()]
    [['dataset', 'model', 'concat_dataset']]
    .reset_index(drop=True)
)

best_concat_cfg.head(10)

In [ ]:
concat_shap = {}  # key: (ds, model_name) -> pd.Series

for _, row in best_concat_cfg.iterrows():
    ds = row['dataset']
    model_name = row['model']
    concat_name = row['concat_dataset']

    print(f'SHAP concat: {ds} / {model_name} / {concat_name}')
    try:
        df_concat, scaler_concat = get_preprocessed_combined(concat_name, concat_name)
        df_test = get_test(ds)
        X_test, _ = align_test_to_train(df_test, df_concat, scaler_concat)

        model = load_concat_model(concat_name, model_name)
        shap_vals = compute_shap(model, X_test)
        importance = mean_abs_shap(shap_vals, X_test.columns)
        concat_shap[(ds, model_name)] = importance.head(TOP_N)
    except Exception as e:
        print(f'SKIP ({e})')
        concat_shap[(ds, model_name)] = None

## Baseline vs Concat (per Dataset × Model)

In [ ]:
for ds in DATASETS:
    n_models = len(MODEL_NAMES)
    fig, axes = plt.subplots(2, n_models, figsize=(n_models * 2.8, 7),
                              sharey=False)

    for col, model_name in enumerate(MODEL_NAMES):
        base_imp = baseline_shap.get((ds, model_name))
        conc_imp = concat_shap.get((ds, model_name))

        for row, (imp, title) in enumerate([
            (base_imp, 'Baseline'),
            (conc_imp, 'Concat')
        ]):
            ax = axes[row][col]
            if imp is not None:
                imp.plot(kind='barh', ax=ax, color='steelblue' if row == 0 else 'darkorange')
                ax.invert_yaxis()
                ax.set_xlabel('mean |SHAP|')
            else:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes)
                ax.axis('off')

            if row == 0:
                ax.set_title(model_name, fontsize=9)
            if col == 0:
                ax.set_ylabel(title, fontsize=10, fontweight='bold')
            else:
                ax.set_ylabel('')

    fig.suptitle(f'Top {TOP_N} Features by SHAP — {ds.upper()}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'result/images/shap_{ds}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print()

## SHAP Table Export

In [ ]:
rows = []
for (ds, model_name), imp in baseline_shap.items():
    if imp is not None:
        for rank, (feat, val) in enumerate(imp.items(), 1):
            rows.append({'dataset': ds, 'model': model_name, 'source': 'baseline',
                         'rank': rank, 'feature': feat, 'mean_abs_shap': val})

for (ds, model_name), imp in concat_shap.items():
    if imp is not None:
        for rank, (feat, val) in enumerate(imp.items(), 1):
            rows.append({'dataset': ds, 'model': model_name, 'source': 'concat',
                         'rank': rank, 'feature': feat, 'mean_abs_shap': val})

df_shap = pd.DataFrame(rows)
df_shap.to_csv('result/shap_importance.csv', index=False)
df_shap.head(10)